# FinExpert v1.1 — Qwen3-4B-Instruct-2507 QLoRA (Unsloth, Colab T4)

**Same training recipe as v1.** This version changes only how the run is *recorded and evaluated*:

| v1 problem | v1.1 fix |
|---|---|
| Two evaluators disagreed (notebook 100% vs repo 93.33% fidelity) | Repo evaluator only; it must score the references at 100% before any model runs |
| Repo cloned at whatever HEAD was current | Exact repo commit recorded in every report |
| Data uploaded by hand, unverified | Data loaded from the repo and checked against SHA-256 hashes |
| Library versions unknown | All versions recorded; `requirements-colab.txt` written for pinning |
| Base-model predictions not saved | Saved and scored with the same evaluator |
| Training history not saved | `train_log.json`, best epoch, loss plot saved |
| Truncated answers undetected | Every answer records its token count and whether it hit the limit |
| Stale markdown | Rewritten |

Everything from one run lands in `runs/<RUN_NAME>/` and is downloaded as one zip.

> Runtime → Change runtime type → **T4 GPU**, then *Run all*. After the run, also do **File → Download → .ipynb** so the saved notebook keeps its outputs.

## 1. Config (only edit this cell)

In [ ]:
# ---- Run identity -------------------------------------------------------
RUN_NAME   = "v1.1-baseline-rerun"
REPO_URL   = "https://github.com/mani1234765/finexpert.git"
REPO_REF   = "main"            # branch, tag or commit SHA; the exact commit used is recorded

# ---- Data (v1 frozen dataset; the run stops if the files differ) --------
EXPECTED_SHA256 = {
    "train":      "b9fe27b8981ed74ef5525852730c4f213578334ea1aec6240b708ad2968baa63",
    "validation": "165c3bf024ef6f2b3f9140116216d570ec4f7ccdb7a0c04e3264a076cc336a84",
    "test":       "851220d99e6d33cfab95334ad3ee9729452860f8aac1a04f5feb22032d5ae33d",
}

# ---- Model + training (identical to v1 — do not change for a v1 rerun) --
MODEL_NAME     = "unsloth/Qwen3-4B-Instruct-2507-unsloth-bnb-4bit"
MAX_SEQ_LENGTH = 2048
SEED           = 3407
LORA_R         = 16
LORA_ALPHA     = 16
LORA_DROPOUT   = 0.0
TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
EPOCHS         = 3
LEARNING_RATE  = 2e-4
BATCH_SIZE     = 2
GRAD_ACCUM     = 4             # effective batch = 8
WARMUP_RATIO   = 0.1
LR_SCHEDULER   = "linear"
WEIGHT_DECAY   = 0.01

# ---- Generation ---------------------------------------------------------
MAX_NEW_TOKENS = 700
GEN_BATCH_SIZE = 6
RUN_BASELINE   = True          # also score the base model (adapter off) on the same test set

# ---- Environment --------------------------------------------------------
UNSLOTH_VERSION = None         # None = latest. After this run, copy the version from run_manifest.json here to pin it.
USE_DRIVE       = False        # True = also copy the run folder to Google Drive
DRIVE_DIR       = "/content/drive/MyDrive/finexpert"
EXPORT_GGUF     = False

RUN_DIR = f"runs/{RUN_NAME}"

## 2. GPU check + install

In [ ]:
import torch
assert torch.cuda.is_available(), "No GPU. Runtime > Change runtime type > T4 GPU"
GPU_NAME = torch.cuda.get_device_name(0)
print(GPU_NAME, "|", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "GB")

In [ ]:
import subprocess, sys
_pkg = "unsloth" if UNSLOTH_VERSION is None else f"unsloth=={UNSLOTH_VERSION}"
_res = subprocess.run([sys.executable, "-m", "pip", "install", "-q", _pkg], capture_output=True, text=True)
if _res.returncode != 0:
    print(_res.stdout[-3000:], _res.stderr[-3000:])
    raise RuntimeError(f"pip install {_pkg} failed")
print("Installed", _pkg)

## 3. Repo + evaluator check

Clones the repo at `REPO_REF`, records the exact commit, and confirms the **fixed** evaluator is present
(the v1 evaluator under-scored correct answers). Then a sanity check: the reference answers scored against
themselves must get 100% on every metric. If the evaluator can't pass that, its numbers on the model can't be trusted.

In [ ]:
import os, subprocess, sys, json, hashlib

REPO_DIR = "finexpert-repo"
os.makedirs(RUN_DIR, exist_ok=True)

def git(*args):
    return subprocess.run(["git", "-C", REPO_DIR, *args], check=True, capture_output=True, text=True).stdout.strip()

if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", "-q", REPO_URL, REPO_DIR], check=True)
git("fetch", "-q", "--all", "--tags")
_is_branch = subprocess.run(["git", "-C", REPO_DIR, "rev-parse", "--verify", "-q", f"origin/{REPO_REF}"],
                            capture_output=True).returncode == 0
git("checkout", "-q", "--detach", f"origin/{REPO_REF}" if _is_branch else REPO_REF)
REPO_COMMIT = git("rev-parse", "HEAD")
assert git("status", "--porcelain") == "", "Repo checkout has local changes; delete finexpert-repo/ and rerun"
print("Repo commit:", REPO_COMMIT)

sys.path.insert(0, os.path.join(REPO_DIR, "src"))
for _m in [m for m in list(sys.modules) if m.startswith("finexpert")]:
    del sys.modules[_m]                      # pick up the checked-out code even when re-running this cell
try:
    from finexpert.evaluation.classification_metrics import classification_metrics  # noqa: F401  (added by the fix)
except ImportError:
    raise RuntimeError(
        "This repo commit has the OLD evaluator. Apply 0001-Fix-evaluator-...patch, push, and rerun."
    )
from finexpert.evaluation.report import build_report, format_report
print("Fixed evaluator present.")

## 4. Load data from the repo (hash-checked)

In [ ]:
from collections import Counter

SPLIT_FILES = {s: os.path.join(REPO_DIR, "data", "sft", f"{s}.jsonl") for s in EXPECTED_SHA256}
DATA_SHA256 = {}
for split, path in SPLIT_FILES.items():
    with open(path, "rb") as f:
        DATA_SHA256[split] = hashlib.sha256(f.read()).hexdigest()
    assert DATA_SHA256[split] == EXPECTED_SHA256[split], (
        f"{split}.jsonl differs from the v1 dataset:\n  got      {DATA_SHA256[split]}\n  expected {EXPECTED_SHA256[split]}"
    )
print("All 3 splits match the v1 dataset hashes.")

def load_split(path):
    raw, flat = [], []
    with open(path, encoding="utf-8") as f:
        for i, line in enumerate(f, 1):
            if not line.strip():
                continue
            r = json.loads(line)
            roles = [m["role"] for m in r["messages"]]
            assert roles == ["system", "user", "assistant"], f"{path} line {i}: roles {roles}"
            d = {m["role"]: m["content"].strip() for m in r["messages"]}
            raw.append(r)
            flat.append({"example_id": r["example_id"], "category": r["category"], "difficulty": r["difficulty"],
                         "system": d["system"], "user": d["user"], "assistant": d["assistant"]})
    return raw, flat

raw, rows = {}, {}
for split, path in SPLIT_FILES.items():
    raw[split], rows[split] = load_split(path)
train_rows, val_rows, test_rows = rows["train"], rows["validation"], rows["test"]
for split, rs in rows.items():
    print(f"{split:10s} n={len(rs):3d}", dict(Counter(r["category"] for r in rs)), dict(Counter(r["difficulty"] for r in rs)))

## 5. Data checks + evaluator self-check

In [ ]:
problems = []
for name, rs in rows.items():
    for r in rs:
        for k in ["example_id", "system", "user", "assistant", "category", "difficulty"]:
            if not str(r[k]).strip():
                problems.append(f"{name}/{r['example_id']}: empty {k}")
        if any(t in r["assistant"] for t in ["<think>", "</think>", "tool_call", "<|im_"]):
            problems.append(f"{name}/{r['example_id']}: template token in answer")
    problems += [f"{name}: duplicate id {i}" for i, n in Counter(r["example_id"] for r in rs).items() if n > 1]
    if len({r["category"] for r in rs}) < 3:
        problems.append(f"{name}: missing a task")

for a, b in [("train", "validation"), ("train", "test"), ("validation", "test")]:
    if {r["user"] for r in rows[a]} & {r["user"] for r in rows[b]}:
        problems.append(f"LEAK: identical prompts in {a} and {b}")
    if {r["example_id"] for r in rows[a]} & {r["example_id"] for r in rows[b]}:
        problems.append(f"LEAK: shared example_ids in {a} and {b}")
assert not problems, "Dataset problems:\n" + "\n".join(problems[:15])

# Evaluator self-check: references vs themselves must be perfect (no model involved).
for split in ["validation", "test"]:
    s = build_report(raw[split], {r["example_id"]: r["messages"][2]["content"] for r in raw[split]})["summary"]
    checks = {
        "figure_fidelity": s["grounding"]["figure_fidelity"],
        "numeric_precision": s["numerical"]["numeric_precision"],
        "numeric_recall": s["numerical"]["numeric_recall"],
        "classification": s["classification"]["accuracy"],
        "section_coverage": s["structure"]["section_coverage"],
    }
    assert all(v == 1.0 for v in checks.values()) and not s["failure_breakdown"]["finding_codes"], (
        f"Evaluator self-check failed on {split}: {checks} {s['failure_breakdown']['finding_codes']}"
    )
print("Data checks passed. Evaluator scores references at 100% on validation and test.")

## 6. Load model + attach LoRA

In [ ]:
from unsloth import FastLanguageModel, is_bfloat16_supported

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME, max_seq_length=MAX_SEQ_LENGTH, dtype=None, load_in_4bit=True,
)
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    target_modules=TARGET_MODULES,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=SEED,
)
model.print_trainable_parameters()

## 7. Format as ChatML + train (loss on the assistant answer only)

ChatML is built by hand so training text and inference prompt match exactly, with no empty `<think>` block.

In [ ]:
from datasets import Dataset

def build_prompt(r):
    return ("<|im_start|>system\n" + r["system"] + "<|im_end|>\n"
            + "<|im_start|>user\n" + r["user"] + "<|im_end|>\n"
            + "<|im_start|>assistant\n")

def to_text(r):
    return build_prompt(r) + r["assistant"] + "<|im_end|>\n"

train_ds = Dataset.from_list([{"text": to_text(r)} for r in train_rows])
val_ds   = Dataset.from_list([{"text": to_text(r)} for r in val_rows])

TOKEN_STATS = {}
for split, rs in rows.items():
    lens = [len(tokenizer(to_text(r)).input_ids) for r in rs]
    ans  = [len(tokenizer(r["assistant"]).input_ids) for r in rs]
    TOKEN_STATS[split] = {"max_total": max(lens), "mean_total": round(sum(lens) / len(lens), 1), "max_answer": max(ans)}
    assert max(lens) <= MAX_SEQ_LENGTH, f"{split}: example longer than MAX_SEQ_LENGTH"
print(TOKEN_STATS)
assert TOKEN_STATS["test"]["max_answer"] < MAX_NEW_TOKENS, "MAX_NEW_TOKENS is below the longest reference answer"
assert all("<think>" not in t["text"] for t in train_ds), "template junk in training text"

In [ ]:
import inspect
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only

_len_key = "max_length" if "max_length" in inspect.signature(SFTConfig.__init__).parameters else "max_seq_length"

args = SFTConfig(
    dataset_text_field="text",
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    num_train_epochs=EPOCHS,
    learning_rate=LEARNING_RATE,
    warmup_ratio=WARMUP_RATIO,
    lr_scheduler_type=LR_SCHEDULER,
    weight_decay=WEIGHT_DECAY,
    optim="adamw_8bit",
    fp16=not is_bfloat16_supported(),
    bf16=is_bfloat16_supported(),
    logging_steps=5,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    seed=SEED,
    output_dir=os.path.join(RUN_DIR, "checkpoints"),
    report_to="none",
    packing=False,
    dataset_num_proc=2,
    **{_len_key: MAX_SEQ_LENGTH},
)

trainer = SFTTrainer(model=model, tokenizer=tokenizer, train_dataset=train_ds, eval_dataset=val_ds, args=args)
trainer = train_on_responses_only(
    trainer, instruction_part="<|im_start|>user\n", response_part="<|im_start|>assistant\n",
)

sample = trainer.train_dataset[0]
labels = [t for t in sample["labels"] if t != -100]
print("Trained tokens in sample 0:", len(labels), "of", len(sample["input_ids"]))
print("Loss starts at:", repr(tokenizer.decode(labels[:25])))
assert "<think>" not in tokenizer.decode(labels), "think block leaked into the training target"
assert "<|im_start|>user" not in tokenizer.decode(labels), "prompt is not masked"

In [ ]:
import time
_t0 = time.time()
stats = trainer.train()
TRAIN_MINUTES = round((time.time() - _t0) / 60, 2)
PEAK_VRAM_GB = round(torch.cuda.max_memory_reserved() / 1e9, 2)

log = trainer.state.log_history
with open(os.path.join(RUN_DIR, "train_log.json"), "w") as f:
    json.dump(log, f, indent=2)

evals = [l for l in log if "eval_loss" in l]
BEST_EVAL_LOSS = trainer.state.best_metric
BEST_EPOCH = next((l["epoch"] for l in evals if l["eval_loss"] == BEST_EVAL_LOSS), None)
print(f"Train time {TRAIN_MINUTES} min | peak VRAM {PEAK_VRAM_GB} GB")
for l in evals:
    print(f"  epoch {l['epoch']:.0f}: val loss {l['eval_loss']:.4f}{'  <- best (loaded)' if l['eval_loss'] == BEST_EVAL_LOSS else ''}")

In [ ]:
import matplotlib.pyplot as plt
tr = [(l["step"], l["loss"]) for l in log if "loss" in l]
ev = [(l["step"], l["eval_loss"]) for l in log if "eval_loss" in l]
plt.figure(figsize=(7, 4))
plt.plot(*zip(*tr), label="train loss")
if ev:
    plt.plot(*zip(*ev), "o-", label="val loss")
plt.xlabel("step"); plt.ylabel("loss"); plt.title(RUN_NAME); plt.legend(); plt.grid(alpha=.3)
plt.savefig(os.path.join(RUN_DIR, "loss_curve.png"), dpi=120, bbox_inches="tight")
plt.show()

## 8. Final evaluation on the held-out TEST set

Test was not used for training or for picking the epoch. Fine-tuned and base model are scored by the **same repo evaluator**.
Every answer records its token count; any answer that hit `MAX_NEW_TOKENS` is flagged, because a cut-off answer would otherwise be scored as if complete.

In [ ]:
FastLanguageModel.for_inference(model)
tokenizer.padding_side = "left"
STOP_IDS = {tokenizer.eos_token_id, tokenizer.convert_tokens_to_ids("<|im_end|>")}

def generate(rows_):
    out = []
    for i in range(0, len(rows_), GEN_BATCH_SIZE):
        batch = rows_[i:i + GEN_BATCH_SIZE]
        enc = tokenizer([build_prompt(r) for r in batch], return_tensors="pt", padding=True).to("cuda")
        with torch.no_grad():
            gen = model.generate(**enc, max_new_tokens=MAX_NEW_TOKENS, do_sample=False,
                                 pad_token_id=tokenizer.pad_token_id if tokenizer.pad_token_id is not None else tokenizer.eos_token_id)
        for r, g in zip(batch, gen):
            new = g[enc["input_ids"].shape[1]:].tolist()
            stop = next((j for j, t in enumerate(new) if t in STOP_IDS), None)
            out.append({
                "example_id": r["example_id"],
                "prediction": tokenizer.decode(new, skip_special_tokens=True).strip(),
                "n_new_tokens": len(new) if stop is None else stop + 1,
                "hit_token_limit": stop is None,
            })
    return out

def save_jsonl(path, items):
    with open(path, "w", encoding="utf-8") as f:
        for it in items:
            f.write(json.dumps(it, ensure_ascii=False) + "\n")

def check_outputs(name, preds_):
    truncated = [p["example_id"] for p in preds_ if p["hit_token_limit"]]
    empty = [p["example_id"] for p in preds_ if not p["prediction"]]
    junk = [p["example_id"] for p in preds_ if "<think>" in p["prediction"] or "tool_call" in p["prediction"]]
    print(f"{name}: truncated={len(truncated)} empty={len(empty)} junk={len(junk)}")
    if truncated:
        print(f"  WARNING: {truncated} hit MAX_NEW_TOKENS; their scores reflect cut-off answers.")
    return {"truncated": truncated, "empty": empty, "junk": junk}

preds = generate(test_rows)
save_jsonl(os.path.join(RUN_DIR, "test_predictions.jsonl"), preds)
OUTPUT_CHECKS = {"finetuned": check_outputs("fine-tuned", preds)}

base_preds = None
if RUN_BASELINE:
    with model.disable_adapter():
        base_preds = generate(test_rows)
    save_jsonl(os.path.join(RUN_DIR, "base_test_predictions.jsonl"), base_preds)
    OUTPUT_CHECKS["base"] = check_outputs("base", base_preds)

In [ ]:
RUN_INFO = {
    "run_name": RUN_NAME,
    "repo_commit": REPO_COMMIT,
    "model_name": MODEL_NAME,
    "data_sha256": DATA_SHA256,
    "max_new_tokens": MAX_NEW_TOKENS,
    "decoding": "greedy",
}

def score(name, preds_, filename):
    report = build_report(raw["test"], {p["example_id"]: p["prediction"] for p in preds_})
    report["run"] = {**RUN_INFO, "predictions": name, "output_checks": OUTPUT_CHECKS[name]}
    with open(os.path.join(RUN_DIR, filename), "w", encoding="utf-8") as f:
        json.dump(report, f, indent=2, ensure_ascii=False)
    with open(os.path.join(RUN_DIR, filename.replace(".json", ".txt")), "w", encoding="utf-8") as f:
        f.write(format_report(report))
    return report

report = score("finetuned", preds, "report.json")
print(format_report(report))

base_report = score("base", base_preds, "base_report.json") if base_preds else None

In [ ]:
def headline(rep):
    s = rep["summary"]
    cm = s["classification_metrics"]
    return {
        "Figure fidelity":       s["grounding"]["figure_fidelity"],
        "Numeric precision":     s["numerical"]["numeric_precision"],
        "Numeric recall":        s["numerical"]["numeric_recall"],
        "Classification acc.":   cm["accuracy"],
        "Macro F1":              cm["macro"]["f1"],
        "Missing labels":        cm["missing_predictions"],
        "Section coverage":      s["structure"]["section_coverage"],
    }

ft = headline(report)
bs = headline(base_report) if base_report else {}
print(f"{'metric':<22}{'base':>10}{'fine-tuned':>12}")
for k, v in ft.items():
    b = bs.get(k)
    fmt = (lambda x: "-" if x is None else (f"{x:.0f}" if k == "Missing labels" else f"{x * 100:.1f}%"))
    print(f"{k:<22}{fmt(b):>10}{fmt(v):>12}")
lo, hi = report["summary"]["classification_metrics"]["accuracy_ci95"]
print(f"\nFine-tuned classification 95% CI: {lo * 100:.1f}-{hi * 100:.1f}% (n={report['summary']['classification_metrics']['n']})")

## 9. Error review (input for the written error analysis)

In [ ]:
by_id = {r["example_id"]: r for r in test_rows}
pred_by_id = {p["example_id"]: p["prediction"] for p in preds}
for e in report["summary"]["failure_breakdown"]["classification_errors"]:
    r = by_id[e["example_id"]]
    print("=" * 100)
    print(f"{e['example_id']} | {r['difficulty']} | expected {e['expected']} -> model {e['generated']}")
    print("INPUT :", r["user"].split("Financial data:")[-1].strip())
    print("MODEL :", pred_by_id[e["example_id"]][:400])

failing = [(ex["example_id"], f["code"]) for ex in report["examples"] for f in ex["findings"] if not f["passed"]]
print("\nAll failing findings:", failing or "none")
print("Verbatim reference matches:", f"{report['summary']['reference_exact_match_rate'] * 100:.1f}%")

## 10. Run manifest + save everything

In [ ]:
import platform, datetime
from importlib.metadata import version, PackageNotFoundError

def v(pkg):
    try:
        return version(pkg)
    except PackageNotFoundError:
        return None

PACKAGES = ["unsloth", "unsloth_zoo", "torch", "transformers", "trl", "peft", "accelerate",
            "bitsandbytes", "datasets", "xformers", "triton"]
VERSIONS = {p: v(p) for p in PACKAGES}

manifest = {
    "run_name": RUN_NAME,
    "finished_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds"),
    "repo_url": REPO_URL,
    "repo_commit": REPO_COMMIT,
    "data_sha256": DATA_SHA256,
    "split_sizes": {s: len(r) for s, r in rows.items()},
    "token_stats": TOKEN_STATS,
    "model_name": MODEL_NAME,
    "training": {
        "seed": SEED, "lora_r": LORA_R, "lora_alpha": LORA_ALPHA, "lora_dropout": LORA_DROPOUT,
        "target_modules": TARGET_MODULES, "epochs": EPOCHS, "learning_rate": LEARNING_RATE,
        "batch_size": BATCH_SIZE, "grad_accum": GRAD_ACCUM, "effective_batch": BATCH_SIZE * GRAD_ACCUM,
        "warmup_ratio": WARMUP_RATIO, "lr_scheduler": LR_SCHEDULER, "weight_decay": WEIGHT_DECAY,
        "optimizer": "adamw_8bit", "precision": "bf16" if is_bfloat16_supported() else "fp16",
        "max_seq_length": MAX_SEQ_LENGTH, "loss_masking": "assistant response only",
        "checkpoint_selection": "best val loss (per epoch)",
    },
    "results": {
        "train_minutes": TRAIN_MINUTES, "peak_vram_gb": PEAK_VRAM_GB,
        "best_eval_loss": BEST_EVAL_LOSS, "best_epoch": BEST_EPOCH,
        "test_finetuned": headline(report),
        "test_base": headline(base_report) if base_report else None,
    },
    "generation": {"max_new_tokens": MAX_NEW_TOKENS, "decoding": "greedy", "output_checks": OUTPUT_CHECKS},
    "environment": {"python": platform.python_version(), "cuda": torch.version.cuda, "gpu": GPU_NAME, "packages": VERSIONS},
}
with open(os.path.join(RUN_DIR, "run_manifest.json"), "w") as f:
    json.dump(manifest, f, indent=2)
with open(os.path.join(RUN_DIR, "requirements-colab.txt"), "w") as f:
    f.write("\n".join(f"{p}=={ver}" for p, ver in VERSIONS.items() if ver) + "\n")

model.save_pretrained(os.path.join(RUN_DIR, "adapter"))       # LoRA adapter only (small)
tokenizer.save_pretrained(os.path.join(RUN_DIR, "adapter"))
print(json.dumps(manifest["results"], indent=2))
print("Packages:", VERSIONS)

In [ ]:
import shutil
shutil.rmtree(os.path.join(RUN_DIR, "checkpoints"), ignore_errors=True)   # large; best weights are already in adapter/
archive = shutil.make_archive(RUN_NAME, "zip", RUN_DIR)
print("Run folder:", sorted(os.listdir(RUN_DIR)))

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    shutil.copytree(RUN_DIR, os.path.join(DRIVE_DIR, "runs", RUN_NAME), dirs_exist_ok=True)
    print("Copied to Drive:", os.path.join(DRIVE_DIR, "runs", RUN_NAME))
else:
    from google.colab import files
    files.download(archive)

In [ ]:
# Optional: merged GGUF for Ollama / llama.cpp (set EXPORT_GGUF=True in the config cell)
if EXPORT_GGUF:
    model.save_pretrained_gguf(os.path.join(RUN_DIR, "gguf"), tokenizer, quantization_method="q4_k_m")

## Reading the results

- **Val loss rising while train loss falls:** overfitting. For a *new* experiment (v2), try `EPOCHS = 2`; don't change it in a v1 rerun.
- **Self-check (section 5) fails:** the evaluator is wrong, not the model. Fix it before trusting any numbers.
- **Classification:** only 10 test examples, so one miss = 10 points. Always quote the 95% CI.
- **Figure fidelity < 100%:** open `report.json` → `INVENTED_FIGURE` findings; check whether the figure is really absent from the input.
- **Verbatim reference match** is a diagnostic. A high rate means the model reproduces the training templates.
- All splits come from the same generator, so this measures learning your format and rules, not general financial skill.
- What to commit to the repo: everything in the zip **except** `adapter/` (adapter weights go to Drive or Hugging Face).